## Homework: Multilingual Embedding-based Machine Translation (6 points)

**In this homework** **<font color='red'>YOU</font>** will make a machine translation system without using parallel corpora, alignment, attention, 100500-depth super-cool recurrent neural networks and all that kind of superstuff.

But even without parallel corpora, this system can be good enough (hopefully). 

For our system we choose two kindred Slavic languages: Ukrainian and Russian. 

### Feel the difference!

(_синій кіт_ vs. _синій кит_)

![blue_cat_blue_whale.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/blue_cat_blue_whale.png)

### Fragment of the Swadesh list for some Slavic languages

The Swadesh list is a lexicostatistical tool. It's named after American linguist Morris Swadesh and contains basic lexis. These lists are used to define subgroupings of languages and their relatedness.

So we can see some kind of word invariance for different Slavic languages.


| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

But the context distribution of these languages demonstrates even more invariance. And we can use this fact for our purposes.

## Data

In [35]:
import gensim
import numpy as np
from gensim.models import KeyedVectors

Download embeddings here:
* [cc.uk.300.vec.zip](https://yadi.sk/d/9CAeNsJiInoyUA)
* [cc.ru.300.vec.zip](https://yadi.sk/d/3yG0-M4M8fypeQ)

Load embeddings for Ukrainian and Russian.

In [36]:
from pathlib import Path
dir_path = Path("/data/bojkoar")


In [37]:
filenames = ["cc.uk.300.vec", "cc.ru.300.vec"]

for filename in filenames:
    full_path = dir_path / (filename)
    !unzip -o {full_path} -d {dir_path}/

Archive:  /data/bojkoar/cc.uk.300.vec
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
Archive:  /data/bojkoar/cc.uk.300.vec.zip
  inflating: /data/bojkoar/cc.uk.300.vec  ^C
Archive:  /data/bojkoar/cc.ru.300.vec
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
Archive:  /data/bojkoar/cc.ru.300.vec.zip
  inflating: /data/bojkoar/cc.ru.300.vec  


In [ ]:
uk_emb: KeyedVectors = KeyedVectors.load_word2vec_format(dir_path / "cc.uk.300.vec")

In [ ]:
ru_emb: KeyedVectors = KeyedVectors.load_word2vec_format(dir_path / "cc.ru.300.vec")

assert uk_emb.vector_size == ru_emb.vector_size == 300

In [ ]:
ru_emb.most_similar([ru_emb["август"]], topn=10)

[('август', 1.0000001192092896),
 ('июль', 0.9383152723312378),
 ('сентябрь', 0.9240029454231262),
 ('июнь', 0.9222574830055237),
 ('октябрь', 0.9095539450645447),
 ('ноябрь', 0.8930036425590515),
 ('апрель', 0.8729087114334106),
 ('декабрь', 0.8652557730674744),
 ('март', 0.8545795679092407),
 ('февраль', 0.8401415944099426)]

In [ ]:
uk_emb.most_similar([uk_emb["серпень"]])

[('серпень', 0.9999998807907104),
 ('липень', 0.9096441268920898),
 ('вересень', 0.9016969203948975),
 ('червень', 0.8992518782615662),
 ('жовтень', 0.8810408115386963),
 ('листопад', 0.8787633180618286),
 ('квітень', 0.8592804670333862),
 ('грудень', 0.8586863279342651),
 ('травень', 0.840811014175415),
 ('лютий', 0.8256431221961975)]

In [ ]:
ru_emb.most_similar([uk_emb["серпень"]])

[('Недопустимость', 0.24435284733772278),
 ('конструктивность', 0.23293082416057587),
 ('офор', 0.23256804049015045),
 ('deteydlya', 0.230317160487175),
 ('пресечении', 0.22632381319999695),
 ('одностороннего', 0.22608886659145355),
 ('подход', 0.2230587750673294),
 ('иболее', 0.22003726661205292),
 ('2015Александр', 0.21872766315937042),
 ('конструктивен', 0.21796567738056183)]

Load small dictionaries for corresponding word pairs as training set and test set.

> **Vocabulary coverage.** The `.vec` files contain vectors for a finite vocabulary only; unlike a full fastText model, `KeyedVectors` loaded from a `.vec` file cannot synthesize vectors for unseen words. Missing words are therefore expected. When loading the bilingual dictionaries, skip a pair if either word is missing from its embedding vocabulary. The loader below reports how many pairs were kept and skipped.

In [ ]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    num_skipped = 0
    with open(filename, "r", encoding="utf-8") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                num_skipped += 1
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    print(f"{filename}: kept {len(uk_ru_pairs)} pairs, skipped {num_skipped} OOV pairs")
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

In [ ]:
uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")

ukr_rus.train.txt: kept 1840 pairs, skipped 87 OOV pairs


In [ ]:
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")

ukr_rus.test.txt: kept 387 pairs, skipped 13 OOV pairs


## Embedding space mapping

Let $x_i \in \mathrm{R}^d$ be the row-vector representation of word $i$ in the source language, and let $y_i \in \mathrm{R}^d$ be the row-vector representation of its translation. Our purpose is to learn a linear transform $W$ that minimizes the squared Euclidean distance between $x_iW$ and $y_i$ for a set of word pairs. Thus we can formulate the following least-squares problem:

$$W^*= \arg\min_W \sum_{i=1}^n||x_iW - y_i||_2^2$$
or
$$W^*= \arg\min_W ||XW - Y||_F^2$$

where the rows of $X$ and $Y$ contain the source and target embeddings and $||*||_F$ is the Frobenius norm.

In Greek mythology, Procrustes or "the stretcher" was a rogue smith and bandit from Attica who attacked people by stretching them or cutting off their legs, so as to force them to fit the size of an iron bed. We do the same bad things with the source embedding space. Our Procrustean bed is the target embedding space.

![embedding_mapping.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

*The figure uses column-vector notation $Wx$. In this notebook embeddings are stored as rows, so the corresponding batched operation is $XW$.*

![procrustes.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

But wait... $W^*= \arg\min_W \sum_{i=1}^n||x_iW - y_i||_2^2$ looks like simple multiple linear regression (without intercept fit). So let's code.

In [ ]:
from sklearn.linear_model import LinearRegression

mapping = LinearRegression(fit_intercept=False)
mapping.fit(X_train, Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float32](300, 300)","[[-0.1 , 0.03, 0.02,..., 0.05, 0.09, 0.02], [-0.04, 0.05, 0.04,..., 0.01, 0.04, 0.03], [ 0.13,-0.01, 0.04,...,-0.02, 0.05,-0.05], ..., [-0.01, 0.01,-0.05,...,-0.04, 0.07, 0. ], [ 0.05, 0.02, 0.03,..., 0.05, 0.07,-0.06], [-0.06,-0.03,-0.04,..., 0.03, 0.02,-0.04]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float,0
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,300
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(300)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float32](300,)","[17.27, 8.14, 7.55,..., 0.65, 0.62, 0.59]"


Let's take a look at neighbors of the vector of word _"серпень"_ (_"август"_ in Russian) after linear transform.

In [ ]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august, topn=10)

[('апрель', 0.8541286587715149),
 ('июнь', 0.8411202430725098),
 ('март', 0.839699387550354),
 ('сентябрь', 0.8359869718551636),
 ('февраль', 0.832929790019989),
 ('октябрь', 0.8311846852302551),
 ('ноябрь', 0.8278923630714417),
 ('июль', 0.823452889919281),
 ('август', 0.8120500445365906),
 ('декабрь', 0.8039003610610962)]

We can see that the neighborhood of this embedding consists of different months, but the right variant is in ninth place.

As a quality measure we will use precision top-1, top-5 and top-10 (for each transformed Ukrainian embedding we count how many correct target pairs are found in the top N nearest neighbors in Russian embedding space).

In [ ]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs = list of correct word pairs [(uk_word_0, ru_word_0), ...]
        mapped_vectors = list of embeddings after mapping from source embedding space to destination embedding space
        topn = the number of nearest neighbors in destination embedding space to choose from
    :returns:
        precision_val, float number, total number of words for which we can find correct translation in top K.
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        options = [w for w, _ in ru_emb.most_similar(mapped_vectors[i], topn=topn)]
        num_matches += (ru in options)
    precision_val = num_matches / len(pairs)
    return precision_val

# contest
from typing import Sequence, Mapping, Tuple

def precision(
        pairs: Sequence[Tuple[str, str]],
        mapped_vectors: np.ndarray,
        embedding_lookup: Mapping[str, np.ndarray],
        topn: int = 1,
    ) -> float:
    if len(pairs) != mapped_vectors.shape[0]: raise ValueError()
    if topn <= 0: raise ValueError()
    if len(pairs) == 0: return 0.0
    items = list(embedding_lookup.items())
    vocab = np.array([word for word, _ in items])
    embedding_vectors = np.array([vector for _, vector in items])
    mapped_units = mapped_vectors / np.linalg.norm(mapped_vectors, axis=1, keepdims=True)
    embedding_units = embedding_vectors / np.linalg.norm(embedding_vectors, axis=1, keepdims=True)
    # cosine distances; shape = len(mapped_vectors) x len(embedding_vectors)
    cos_dists = mapped_units @ embedding_units.T
    most_similar = np.argsort(-cos_dists, axis=1, kind="stable")[:, :topn]
    pairs_arr = np.array([ru for _, ru in pairs])
    matches = np.any(pairs_arr[:, None] == vocab[most_similar], axis=1)
    return float(np.mean(matches))

In [ ]:
precision(
    [(0, 'long_horiz'), (1, 'downward'), (2, 'long_horiz'), (3, 'diag')],
    np.array([[1, 0], [0, 1], [3, 2], [2, 3]]),
    {'long_horiz': [2, 0], 'diag': [1, 1], 'downward': [0, -1]},
    topn=2
)

0.75

In [ ]:
precision(
    [],
    np.zeros([0, 2]),
    {'long_horiz': [2, 0], 'diag': [1, 1], 'downward': [0, -1]},
    topn=2
)

0.0

In [ ]:
contest_embedding_lookup = {
    key: ru_emb[key] for key in ru_emb.key_to_index
}

In [ ]:
assert precision([("серпень", "август")], august, contest_embedding_lookup, topn=5) == 0.0
assert precision([("серпень", "август")], august, contest_embedding_lookup, topn=9) == 1.0
assert precision([("серпень", "август")], august, contest_embedding_lookup, topn=10) == 1.0

In [ ]:
print(precision(uk_ru_test, X_test, contest_embedding_lookup))
assert precision(uk_ru_test, X_test, contest_embedding_lookup) == 0.0
assert precision(uk_ru_test, Y_test, contest_embedding_lookup) == 1.0

0.0


In [ ]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), contest_embedding_lookup, 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), contest_embedding_lookup, 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.813

## Making it better (orthogonal Procrustean problem)

It can be shown (see [original paper](https://arxiv.org/pdf/1710.04087)) that a self-consistent linear mapping between semantic spaces should be orthogonal. 
We can restrict the transform $W$ to be orthogonal. Then we will solve the next problem:

$$W^*= \arg\min_W ||XW - Y||_F^2 \text{, where: } W^TW = I$$

$$I \text{ - identity matrix}$$

Instead of making yet another regression problem, we can find the optimal orthogonal transformation using singular value decomposition. It turns out that the optimal transformation $W^*$ can be expressed via SVD components:
$$X^TY=U\Sigma V^T\text{, singular value decomposition}$$
$$W^*=UV^T$$

In [ ]:
def learn_transform(X_train: np.ndarray, Y_train: np.ndarray):
    """ 
    :returns: W* : float matrix[emb_dim x emb_dim] as defined in formulae above
    """
    u, _, vh = np.linalg.svd(X_train.T @ Y_train)
    return u @ vh

In [ ]:
W = learn_transform(X_train, Y_train)

In [ ]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])

[('апрель', 0.8237906694412231),
 ('сентябрь', 0.8049712181091309),
 ('март', 0.802565336227417),
 ('июнь', 0.8021842241287231),
 ('октябрь', 0.8001735210418701),
 ('ноябрь', 0.793448269367218),
 ('февраль', 0.7914120554924011),
 ('июль', 0.7908109426498413),
 ('август', 0.7891016006469727),
 ('декабрь', 0.7686371803283691)]

In [ ]:
assert precision(uk_ru_test, np.matmul(X_test, W), contest_embedding_lookup) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), contest_embedding_lookup, 5) >= 0.824

In [ ]:
precision([('серпень', 'апрель')], np.matmul([uk_emb['серпень']], W), contest_embedding_lookup)

1.0

## UK-RU Translator

Now we are ready to make a simple word-based translator: for each word in the source language in shared embedding space we find the nearest in the target language.

### Out-of-vocabulary policy

`translate` receives a lowercased sentence and processes its whitespace-separated tokens (`sentence.split()`). For each token:
* if it contains no letters (`any(char.isalpha() for char in token)` is false), copy it unchanged regardless of embedding-vocabulary membership;
* otherwise, if it is present in `uk_emb`, translate it using the learned mapping;
* otherwise, output the literal string `<UNK>`.

`<UNK>` is an output marker and does not need to have an embedding. The embedding vocabulary is finite, so unknown Ukrainian words are expected and must not cause a `KeyError` or be copied unchanged. The supplied fairy tale is not re-tokenized for this task: punctuation attached to a word remains part of that whitespace token and the whole token follows the OOV rule. Seeing several `<UNK>` tokens in the final translation is therefore expected. These rules are part of the grading contract.

In [38]:
with open("fairy_tale.txt", "r", encoding="utf-8") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]

In [ ]:
def translate(sentence):
    """
    :args:
        sentence - a lowercased sentence in Ukrainian (str)
    :returns:
        translation - sentence in Russian (str)

    * copy punctuation and numbers unchanged
    * translate known lexical tokens through the learned mapping
    * replace unknown lexical tokens with <UNK>
    """
    

# contest

def translate(
        sentence: str,
        source_embeddings: Mapping[str, np.ndarray],
        target_embeddings: Mapping[str, np.ndarray],
        transform: np.ndarray
    ) -> str:
    result = []
    for token in sentence.split():
        source_vector = source_embeddings.get(token)
        if source_vector is None:
            result.append(token)
            continue
        source_unit = source_vector @ transform / np.linalg.norm(source_vector)
        most_similar_token = None
        closest_dist = -1.0
        for target_token, target_vector in target_embeddings.items():
            target_unit = target_vector / np.linalg.norm(target_vector)
            cos_dist = (source_unit @ target_unit.T).item()
            if cos_dist > closest_dist:
                most_similar_token = target_token
                closest_dist = cos_dist
        result.append(most_similar_token)
    return " ".join(result)

In [46]:
uk_embedding_lookup = {
    key: uk_emb[key] for key in uk_emb.key_to_index
}
ru_embedding_lookup = {
    key: ru_emb[key] for key in ru_emb.key_to_index
}
args = [uk_embedding_lookup, ru_embedding_lookup, W] #, True]

print((np.array([uk_embedding_lookup['серпень'], uk_embedding_lookup['липень']]) @ W)[:5, :5])
print(np.array([uk_emb['серпень'] @ W, uk_emb['липень'] @ W])[:5, :5])

[[-0.00734629 -0.04676873 -0.03212409 -0.02578139  0.01993696]
 [ 0.00883744 -0.05114774 -0.02656783 -0.01307035  0.0066713 ]]
[[-0.00734629 -0.04676873 -0.03212408 -0.02578139  0.01993696]
 [ 0.00883744 -0.05114773 -0.02656783 -0.01307033  0.0066713 ]]


In [47]:
translate('серпень', *args)

'апрель'

In [48]:
oov_word = "цьогословаточнонемаєвсловнику"
assert oov_word not in uk_emb
assert translate(oov_word, *args) == oov_word #"<UNK>"
assert translate(f"кіт {oov_word} !", *args) == f"кот {oov_word} !" #"кот <UNK> !"
assert translate(".", *args) == "."
assert translate("1 , 3", *args) == "1 , 3"
assert translate("кіт зловив мишу", *args) == "кот поймал мышку"

In [49]:
print(translate("кіт , ти маму мав ?", *args))
print(translate("а пахне як !", *args))
print(translate("за сорок гривень штани , блять !", *args))

кот , ты маму имел ?
а пахнет как !
за сорок гривен штаны , падлу !


In [50]:
import re
for sentence in uk_sentences:
    sentence = re.sub(r"\b([\.,!?:;]|\.\.\.)", r" \1", sentence)
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence, *args)))

src: лисичка - сестричка і вовк - панібрат
dst: лисичка – сестричка и волк – панібрат



KeyboardInterrupt: 

Not so bad, right? We can easily improve translation using a language model and not one but several nearest neighbors in shared embedding space. But that's for next time.

## Would you like to learn more?

### Articles:
* [Exploiting Similarities among Languages for Machine Translation](https://arxiv.org/pdf/1309.4168)  - entry point for multilingual embedding studies by Tomas Mikolov (the author of W2V)
* [Offline bilingual word vectors, orthogonal transformations and the inverted softmax](https://arxiv.org/pdf/1702.03859) - orthogonal transform for unsupervised MT
* [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087)
* [Loss in Translation: Learning Bilingual Word Mapping with a Retrieval Criterion](https://arxiv.org/pdf/1804.07745)
* [Unsupervised Alignment of Embeddings with Wasserstein Procrustes](https://arxiv.org/pdf/1805.11222)

### Repos (with ready-to-use multilingual embeddings):
* https://github.com/facebookresearch/MUSE

* https://github.com/Babylonpartners/fastText_multilingual -